# Notebook 3: Agent Tools, DB State & `LLM-as-a-Judge` (**DeepEval**)

ඔයාට කලින් පැටලුණු **`Golden Dataset` vs `Agent` vs `LLM-as-a-Judge`** වෙනස මෙතනින් 100% පැහැදිලි වෙනවා:
1. **Part A — Agent Tools & DB State (`No LLM Judge needed!`):** Agent හරි tools call කළාද + DB එක update වුණාද කියලා DeepEval `ToolCorrectnessMetric` සහ `ORDERS_DB` වලින් code එකෙන් කෙලින්ම check කරනවා.
2. **Part B — `LLM-as-a-Judge` (`DeepEval GEval`) + Human Calibration:** *"Tone එක polite ද?"* වගේ subjective දෙයක් විතරක් Judge කෙනෙක්ට දීලා, Human labels එක්ක **`85%+` Agreement** තියෙනවද බලනවා.
3. **Part C — `pass@k` vs `pass^k`:** Trial 5ක් ඇත්තටම simulate කරලා consistency difference එක බලමු!

In [1]:
import pandas as pd
import deepeval
from deepeval.metrics import GEval, ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, SingleTurnParams, ToolCall
import sys
if ".." not in sys.path:
    sys.path.append("..")

from lankashop import ORDERS_DB, ask_agent, get_deepeval_judge, reset_db, set_live_mode

USE_LIVE_API = False  # Live OpenAI GEval Judge සඳහා True කරන්න
set_live_mode(USE_LIVE_API)
print("DeepEval version:", deepeval.__version__)

Active Mode: MOCK ($0 offline) | OpenAI Key in .env: True
DeepEval version: 4.3.3


## Part A: Agent Tools & Backend DB State (`LLM Judge ඕන නෑ!`)

In [3]:
tool_metric = ToolCorrectnessMetric(threshold=1.0, should_consider_ordering=True, include_reason=False, async_mode=False)

agent_cases = [
    ("Valid Refund (ORD-100)", "Please refund my unopened laptop order ORD-100.", "none", ["get_order", "issue_refund"], "ORD-100", True),
    ("Denied Earbuds (ORD-200)", "Please refund my opened Wireless Earbuds order ORD-200.", "none", ["get_order"], "ORD-200", False),
    ("BUGGY AGENT (Skips get_order!)", "Please refund my opened Wireless Earbuds order ORD-200.", "bad_tool", ["get_order"], "ORD-200", False),
]

tool_rows = []
for name, query, bm, expected_tools, order_id, expected_refunded in agent_cases:
    reset_db()
    tr = ask_agent(query, bug_mode=bm)

    tc = LLMTestCase(
        input=query,
        actual_output=tr["answer"],
        tools_called=[ToolCall(name=t["name"], input_parameters=t["args"]) for t in tr["tools_called"]],
        expected_tools=[ToolCall(name=n) for n in expected_tools],
    )
    tool_metric.measure(tc)

    tool_rows.append({
        "case": name,
        "tools_called": [t["name"] for t in tr["tools_called"]],
        "tool_pass": tool_metric.is_successful(),
        "db_refunded": ORDERS_DB[order_id]["refunded"],
        "db_state_ok": ORDERS_DB[order_id]["refunded"] is expected_refunded,
    })

pd.DataFrame(tool_rows)

Output()

Output()

Output()

,case,tools_called,tool_pass,db_refunded,db_state_ok
0,Valid Refund (ORD-100),"[get_order, issue_refund]",True,True,True
1,Denied Earbuds (ORD-200),[get_order],True,False,True
2,BUGGY AGENT (Skips get_order!),[issue_refund],False,True,False


## Part B: `LLM-as-a-Judge` (`DeepEval GEval`) & Human Calibration
මෙතන Roles 3 වෙනම බලන්න:
- `query` ➔ **LankaShop Agent (ශිෂ්‍යයා)** උත්තරයක් හදනවා (`answer`).
- **`GEval` (ගුරුවරයා / Judge)** ඒ answer එක polite ද බලලා PASS (`1`) / FAIL (`0`) දෙනවා.
- **`human_label` (Human Reviewer)** එක්ක compare කරලා Judge විශ්වාස කරන්න පුළුවන්ද (`>= 85%`) බලනවා!

In [4]:
geval = GEval(
    name="Polite Support",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=["Check tone is polite and never insults the customer."],
    threshold=0.7,
    model=get_deepeval_judge(),
    async_mode=False,
)

labeled_samples = [
    ("Polite Refund", "Please refund my unopened laptop order ORD-100.", "none", 1),
    ("Polite Denial", "Please refund my opened Wireless Earbuds order ORD-200.", "none", 1),
    ("Rude Bug", "Please refund my opened Wireless Earbuds order ORD-200.", "rude_tone", 0),
]

judge_rows = []
for name, query, bm, human_pass in labeled_samples:
    reset_db()
    agent_answer = ask_agent(query, bug_mode=bm)["answer"]
    geval.measure(LLMTestCase(input=query, actual_output=agent_answer))
    judge_pass = int(geval.is_successful())

    judge_rows.append({
        "case": name,
        "agent_answer": agent_answer[:45] + "...",
        "judge_pass": judge_pass,
        "human_pass": human_pass,
        "agreed": judge_pass == human_pass,
    })

df_j = pd.DataFrame(judge_rows)
print(f"Judge-Human Agreement: {df_j['agreed'].mean():.1%} (Production Target >= 85%)")
df_j

Output()

Output()

Output()

Judge-Human Agreement: 100.0% (Production Target >= 85%)


,case,agent_answer,judge_pass,human_pass,agreed
0,Polite Refund,Verified ORD-100 and processed your full refu...,1,1,True
1,Polite Denial,"Sorry, ORD-200 cannot be refunded because ope...",1,1,True
2,Rude Bug,Stop wasting support time! Read the website y...,0,0,True


## Part C: `pass@k` vs `pass^k` (Agent Consistency)
එක try එකක success rate එක `90%` (`0.90`) වුණොත්, **5 පාරක්** run කරද්දී වෙන දේ ඇත්තටම simulate කරලා බලමු:

In [5]:
import random

random.seed(42)
p, k, episodes = 0.90, 5, 1000
at_least_one, all_five = 0, 0

for _ in range(episodes):
    trials = [random.random() < p for _ in range(k)]
    at_least_one += int(any(trials))
    all_five += int(all(trials))

print(f"Simulated pass@5 (at least 1 ok): {at_least_one / episodes:.1%}")
print(f"Simulated pass^5 (ALL 5 ok!):     {all_five / episodes:.1%}  <- Refund Agent එකකට ඕන මේක!")

Simulated pass@5 (at least 1 ok): 100.0%
Simulated pass^5 (ALL 5 ok!):     59.9%  <- Refund Agent එකකට ඕන මේක!
